In [ ]:
# xử lí dataset để train

In [ ]:
import argparse
from pathlib import Path
import numpy as np
import sys

def reshape_labels(labels_flat: np.ndarray) -> np.ndarray:
    """(N, 57) → (N, 19, 3).
    Input layout:  [x0..x18, y0..y18, z0..z18]
    Output layout: joints[i] = [xi, yi, zi]
    """
    N = labels_flat.shape[0]
    return labels_flat.reshape(N, 3, 19).transpose(0, 2, 1).astype(np.float32)

def compute_norm_stats(X_train: np.ndarray, Y_train: np.ndarray) -> dict:
    """Compute normalization statistics from TRAIN split only."""
    C = X_train.shape[-1]
    flat = X_train.reshape(-1, C)
    feat_low = np.percentile(flat, 1, axis=0).astype(np.float32)
    feat_high = np.percentile(flat, 99, axis=0).astype(np.float32)

    pj_mins = Y_train.min(axis=0).astype(np.float32)   # (19, 3)
    pj_maxs = Y_train.max(axis=0).astype(np.float32)   # (19, 3)

    return {
        "feat_low": feat_low,
        "feat_high": feat_high,
        "pj_mins": pj_mins,
        "pj_maxs": pj_maxs,
    }

def normalize_features(X: np.ndarray, low: np.ndarray, high: np.ndarray) -> np.ndarray:
    """Per-channel percentile normalize → [-1, +1]. Outliers clipped."""
    range_each = high - low + 1e-6
    out = 2.0 * (X - low) / range_each - 1.0
    return np.clip(out, -1.0, 1.0).astype(np.float32)

def normalize_labels(Y: np.ndarray, mins: np.ndarray, maxs: np.ndarray) -> np.ndarray:
    """Per-joint per-axis normalize → [-1, +1]."""
    range_each = (maxs - mins).clip(min=1e-6)
    out = 2.0 * (Y - mins) / range_each - 1.0
    return np.clip(out, -1.0, 1.0).astype(np.float32)

def main():
    # Fix cho Kaggle Notebook: Sử dụng thư mục mặc định nếu chạy trong Cell
    if 'ipykernel' in sys.modules:
        raw_dir = Path("/kaggle/input/datasets/thanhqucl/mars-data")
        # LƯU Ý: Nếu Kaggle báo không tìm thấy file, hãy thử đổi raw_dir thành:
        # raw_dir = Path("/kaggle/input/mars-data") 
        out_dir = Path("/kaggle/working/data/processed/mars")
    else:
        # Chạy bằng Terminal
        parser = argparse.ArgumentParser(description="Preprocess MARS dataset for eMamba")
        parser.add_argument("--raw_dir", type=str, default="/kaggle/input/datasets/thanhqucl/mars-data")
        parser.add_argument("--out_dir", type=str, default="/kaggle/working/data/processed/mars")
        args = parser.parse_args()
        raw_dir = Path(args.raw_dir)
        out_dir = Path(args.out_dir)

    out_dir.mkdir(parents=True, exist_ok=True)

    # ── Load raw .npy ──────────────────────────────────────────────────
    print(f"Loading raw MARS data from {raw_dir}...")
    splits = {
        "train":    ("featuremap_train.npy",    "labels_train.npy"),
        "val":      ("featuremap_validate.npy", "labels_validate.npy"),
        "test":     ("featuremap_test.npy",     "labels_test.npy"),
    }

    data = {}
    for split_name, (feat_file, label_file) in splits.items():
        try:
            X = np.load(raw_dir / feat_file).astype(np.float32)
            Y_flat = np.load(raw_dir / label_file).astype(np.float32)
        except FileNotFoundError as e:
            print(f"❌ LỖI: Không tìm thấy file {e.filename}. Hãy kiểm tra lại đường dẫn raw_dir!")
            return

        Y = reshape_labels(Y_flat)  # (N, 57) → (N, 19, 3)
        data[split_name] = {"X": X, "Y": Y}
        print(f"  {split_name}: X={X.shape}, Y={Y.shape}")

    total = sum(d["X"].shape[0] for d in data.values())
    print(f"  Total: {total} frames (paper: 40,083)")

    # ── Compute normalization stats (TRAIN only) ───────────────────────
    print("\nComputing normalization stats from train split...")
    stats = compute_norm_stats(data["train"]["X"], data["train"]["Y"])

    print(f"  Feature bounds per channel:")
    for c in range(stats["feat_low"].shape[0]):
        print(f"    ch{c}: [{stats['feat_low'][c]:.3f}, {stats['feat_high'][c]:.3f}]")

    print(f"  Label ranges (meters):")
    for axis, name in enumerate(["X", "Y", "Z"]):
        lo = stats["pj_mins"][:, axis]
        hi = stats["pj_maxs"][:, axis]
        print(f"    {name}: min_joint_range={lo.min():.3f}m, max_joint_range={hi.max():.3f}m")

    # ── Normalize all splits ───────────────────────────────────────────
    print("\nNormalizing...")
    for split_name in data:
        data[split_name]["X"] = normalize_features(
            data[split_name]["X"], stats["feat_low"], stats["feat_high"]
        )
        data[split_name]["Y"] = normalize_labels(
            data[split_name]["Y"], stats["pj_mins"], stats["pj_maxs"]
        )

    # Verify ranges
    for split_name in data:
        X, Y = data[split_name]["X"], data[split_name]["Y"]
        print(f"  {split_name}: X∈[{X.min():.2f}, {X.max():.2f}], Y∈[{Y.min():.2f}, {Y.max():.2f}]")

    # ── Save .npz ──────────────────────────────────────────────────────
    print(f"\nSaving to {out_dir}/...")

    for split_name in data:
        X = data[split_name]["X"]
        Y = data[split_name]["Y"]
        frame_ids = np.arange(X.shape[0], dtype=np.int64)
        out_path = out_dir / f"{split_name}.npz"
        np.savez(out_path, X=X, Y=Y, frame_ids=frame_ids)
        size_mb = out_path.stat().st_size / 1024 / 1024
        print(f"  {out_path.name}: {X.shape[0]} frames, {size_mb:.1f} MB")

    # Save norm stats (needed for denormalize during eval → cm)
    stats_path = out_dir / "norm_range.npz"
    np.savez(stats_path, **stats)
    print(f"  norm_range.npz: saved (feat_low/high, pj_mins/maxs)")

    print("\nDone! Ready for training.")

if __name__ == "__main__":
    main()

In [ ]:
#train float 32

In [ ]:
cd /kaggle/input/datasets/thanhqucl/code-fp-32

In [ ]:
#!/usr/bin/env python3
"""
train_fp32_penalty.py — Train FP32 với SSM output penalty
==========================================================
Phương án C: thêm regularization ép SSM output nhỏ (< 3.5)
để khi convert INT8 ít bị clip.

Dùng với FP32 model code (KHÔNG phải INT8).
"""

import os, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader

from model.mamba_model import build_model
from train import MambaConfig

# ── SSM penalty config ──
SSM_PENALTY_WEIGHT = 0.1    # Hệ số phạt (0.05-0.2, tune)
SSM_THRESHOLD      = 3.5    # Ngưỡng: phạt nếu |y_ssm| > 3.5 (Q5 max=4.0)

class MARSDataset(Dataset):
    def __init__(self, X, Y):
        self.x = torch.from_numpy(X.astype(np.float32))
        self.y = torch.from_numpy(Y.astype(np.float32))
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i]

def preprocess():
    base = "/kaggle/working/data/processed/mars"
    train_data = np.load(f"{base}/train.npz")
    test_data  = np.load(f"{base}/test.npz")
    X_tr = train_data['X'].astype(np.float32)
    Y_tr = train_data['Y'].astype(np.float32).reshape(-1, 57)
    X_te = test_data['X'].astype(np.float32)
    Y_te = test_data['Y'].astype(np.float32).reshape(-1, 57)
    nr = np.load(f"{base}/norm_range.npz")
    return X_tr, Y_tr, X_te, Y_te, nr['pj_mins'], nr['pj_maxs']

def evaluate(model, loader, device, pj_mins, pj_maxs):
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for x, y in loader:
            preds.append(model(x.to(device)).cpu())
            gts.append(y)
    pred_cm = ((torch.cat(preds).numpy().reshape(-1,19,3) + 1)/2 * (pj_maxs-pj_mins) + pj_mins) * 100
    gt_cm   = ((torch.cat(gts).numpy().reshape(-1,19,3) + 1)/2 * (pj_maxs-pj_mins) + pj_mins) * 100
    mae   = np.mean(np.abs(pred_cm - gt_cm))
    rmse  = np.sqrt(np.mean((pred_cm - gt_cm)**2))
    mpjpe = np.mean(np.linalg.norm(pred_cm - gt_cm, axis=-1))
    return dict(mae_cm=mae, rmse_cm=rmse, mpjpe_cm=mpjpe)


class SSMPenaltyTrainer:
    """Bắt SSM output qua hooks, tính penalty."""
    def __init__(self, model, threshold=SSM_THRESHOLD):
        self.model = model
        self.threshold = threshold
        self.ssm_outputs = []
        self.hooks = []

        for blk in model.blocks:
            h = blk.ssm.register_forward_hook(self._make_hook())
            self.hooks.append(h)

    def _make_hook(self):
        def fn(module, inp, out):
            self.ssm_outputs.append(out)
        return fn

    def compute_penalty(self):
        penalty = torch.tensor(0.0, device=next(self.model.parameters()).device)
        for out in self.ssm_outputs:
            # Phạt phần vượt ngưỡng: mean(relu(|y| - threshold))
            excess = torch.relu(out.abs() - self.threshold)
            penalty = penalty + excess.mean()
        self.ssm_outputs.clear()
        return penalty

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()


def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"SSM penalty: weight={SSM_PENALTY_WEIGHT}, threshold={SSM_THRESHOLD}\n")

    cfg = MambaConfig()
    X_tr, Y_tr, X_te, Y_te, pj_mins, pj_maxs = preprocess()

    kw = dict(num_workers=2, pin_memory=True)
    train_loader = DataLoader(MARSDataset(X_tr, Y_tr), batch_size=cfg.batch_size, shuffle=True, **kw)
    test_loader  = DataLoader(MARSDataset(X_te, Y_te), batch_size=cfg.batch_size, shuffle=False, **kw)

    model = build_model(cfg).to(device)
    model.summary()

    trainer = SSMPenaltyTrainer(model, threshold=SSM_THRESHOLD)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.MSELoss()

    best_mpjpe = float('inf')
    warmup = 10
    cosine_epochs = 40

    for epoch in range(cfg.epochs):
        # LR scheduler
        if epoch < warmup:
            lr = 1e-3 * (epoch + 1) / warmup
        elif epoch < cfg.epochs - cosine_epochs:
            lr = 1e-3
        else:
            t = (epoch - (cfg.epochs - cosine_epochs)) / cosine_epochs
            lr = 1e-3 * 0.5 * (1 + math.cos(math.pi * t))
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        model.train()
        total_loss = 0.0
        total_penalty = 0.0
        start = time.time()

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            pred = model(x)
            mse_loss = criterion(pred, y)
            ssm_penalty = trainer.compute_penalty()
            loss = mse_loss + SSM_PENALTY_WEIGHT * ssm_penalty

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += mse_loss.item()
            total_penalty += ssm_penalty.item()

        metrics = evaluate(model, test_loader, device, pj_mins, pj_maxs)
        t_ep = time.time() - start

        # Monitor SSM output range
        model.eval()
        ssm_maxes = []
        with torch.no_grad():
            sample_x = torch.from_numpy(X_te[:1]).float().to(device)
            _ = model(sample_x)
            for out in trainer.ssm_outputs:
                ssm_maxes.append(out.abs().max().item())
            trainer.ssm_outputs.clear()

        ssm_max_str = "/".join([f"{m:.1f}" for m in ssm_maxes])

        print(f"Ep {epoch+1:3d}/{cfg.epochs} | lr={lr:.1e} | MSE={total_loss:.4f} | "
              f"Penalty={total_penalty:.4f} | MAE={metrics['mae_cm']:.3f} | "
              f"RMSE={metrics['rmse_cm']:.3f} | MPJPE={metrics['mpjpe_cm']:.3f} | "
              f"SSM_max={ssm_max_str} | {t_ep:.1f}s")

        if metrics['mpjpe_cm'] < best_mpjpe:
            best_mpjpe = metrics['mpjpe_cm']
            save_path = "/kaggle/working/best_fp32.pt"
            torch.save({"model_state_dict": model.state_dict()}, save_path)
            print(f"  → Best! Saved {save_path} (MPJPE: {best_mpjpe:.3f} cm)")

    trainer.remove_hooks()
    print(f"\n✅ Done! Best MPJPE: {best_mpjpe:.3f} cm")
    print(f"   SSM output target: < {SSM_THRESHOLD}")

if __name__ == "__main__":
    main()

In [ ]:
# train int8

In [ ]:
cd /kaggle/input/code-int8

In [ ]:
#!/usr/bin/env python3
"""
train.py — QAT Warm-Start từ FP32 (PER-LAYER FRAC + INT16 SSM)
===============================================================
FIX đầy đủ:
  ✓ clamp_weights PER-LAYER FRAC (khớp BLOCK_FRAC)
  ✓ delta_2.bias INT16 (±32767/256)
  ✓ reset SSM state trong train + eval (tránh state leak)
  ✓ warm-start từ best_fp32, lr=2e-5, không warmup (tránh diverge)
"""

import os, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader

from model.mamba_model import build_model, BLOCK_FRAC, HEAD_FRAC, PATCH_FRAC
from model.ssm import SCALE, A_SCALE, D2_BIAS_SCALE, FRAC


@dataclass
class MambaConfig:
    input_height: int = 8
    input_width: int = 8
    in_channels: int = 5
    d_model: int = 20
    expand: int = 2
    d_state: int = 8
    d_conv: int = 4
    n_mamba_blocks: int = 2
    patch_size: int = 2
    dt_rank: int = 3
    norm_type: str = "range_norm"
    softplus_type: str = "relu"
    silu_type: str = "silu"
    output_dim: int = 57
    batch_size: int = 32

    epochs: int = 60
    lr: float = 2e-5              # warm-start LR cực nhỏ (tránh diverge)
    weight_decay: float = 1e-4
    warmup_epochs: int = 0        # KHÔNG warmup (warmup lên LR cao gây diverge)
    grad_clip: float = 0.5

    fp32_ckpt: str = "/kaggle/input/datasets/thanhqucl/file-pth/best_fp32 (2).pt"

    @property
    def d_inner(self): return self.expand * self.d_model
    @property
    def seq_len(self):
        return (self.input_height // self.patch_size) * (self.input_width // self.patch_size)
    @property
    def patch_dim(self):
        return self.patch_size * self.patch_size * self.in_channels

    def print_summary(self):
        print("=" * 75)
        print("eMamba QAT — Per-layer FRAC + INT16 SSM + Warm-start")
        print("=" * 75)
        print(f"  D={self.d_model} | E={self.expand} | D_INNER={self.d_inner}")
        print(f"  N_STATE={self.d_state} | BLOCKS={self.n_mamba_blocks} | SEQ_LEN={self.seq_len}")
        print(f"  Epochs={self.epochs} | LR={self.lr} | Warmup={self.warmup_epochs} | clip={self.grad_clip}")
        print(f"  Warm-start: {self.fp32_ckpt if self.fp32_ckpt else 'NONE (from scratch)'}")
        print("=" * 75)


# ── Per-layer FRAC map ──
def build_frac_map(cfg):
    fm = {}
    pf = PATCH_FRAC
    fm['embedding.proj.weight'] = pf['w_frac']
    fm['embedding.proj.bias']   = pf.get('bias_frac', pf['w_frac'])
    for i in range(cfg.n_mamba_blocks):
        bf = BLOCK_FRAC[i]; p = f'blocks.{i}.'
        fm[p+'norm.gamma'] = FRAC
        fm[p+'norm.beta']  = FRAC
        fm[p+'proj_x.weight'] = bf['frac_proj_x']
        fm[p+'proj_z.weight'] = bf['frac_proj_z']
        fm[p+'conv1d.weight'] = bf['frac_conv_w']
        fm[p+'conv1d.bias']   = bf.get('frac_conv_b', bf['frac_conv_w'])
        fm[p+'ssm.selection.proj_b.weight']       = bf['frac_sel_b']
        fm[p+'ssm.selection.proj_c.weight']       = bf['frac_sel_c']
        fm[p+'ssm.selection.proj_delta_1.weight'] = bf['frac_sel_d1']
        fm[p+'ssm.selection.proj_delta_2.weight'] = bf['frac_sel_d2']
        fm[p+'ssm.D'] = FRAC
        fm[p+'out_proj.weight'] = bf['frac_outproj']
    fm['norm.gamma'] = FRAC
    fm['norm.beta']  = FRAC
    hf = HEAD_FRAC
    fm['head.ffn_in.weight']  = hf['frac_ffn_in_w']
    fm['head.ffn_in.bias']    = hf['frac_ffn_in_b']
    fm['head.ffn_out.weight'] = hf['frac_ffn_out_w']
    fm['head.ffn_out.bias']   = hf['frac_ffn_out_b']
    fm['head.proj.weight']    = hf['frac_proj_w']
    fm['head.proj.bias']      = hf['frac_proj_b']
    return fm

FRAC_MAP = None


def clamp_weights(model):
    with torch.no_grad():
        for name, p in model.named_parameters():
            if 'A_log' in name:
                p.clamp_(max=math.log(128.0 / float(A_SCALE)))
            elif 'proj_delta_2.bias' in name:
                p.clamp_(-32768.0 / float(D2_BIAS_SCALE),
                          32767.0 / float(D2_BIAS_SCALE))   # INT16
            else:
                f = FRAC_MAP.get(name, FRAC) if FRAC_MAP else FRAC
                scale = 1 << f
                p.clamp_(-128.0 / scale, 127.0 / scale)     # per-layer FRAC


def reset_ssm(model):
    for blk in model.blocks:
        if hasattr(blk.ssm, 'reset_state'):
            blk.ssm.reset_state()


class MARSDataset(Dataset):
    def __init__(self, X, Y):
        self.x = torch.from_numpy(X.astype(np.float32))
        self.y = torch.from_numpy(Y.astype(np.float32))
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i]


def preprocess():
    base = "/kaggle/working/data/processed/mars"
    tr = np.load(f"{base}/train.npz"); te = np.load(f"{base}/test.npz")
    X_tr = tr['X'].astype(np.float32); Y_tr = tr['Y'].astype(np.float32).reshape(-1, 57)
    X_te = te['X'].astype(np.float32); Y_te = te['Y'].astype(np.float32).reshape(-1, 57)
    nr = np.load(f"{base}/norm_range.npz")
    return X_tr, Y_tr, X_te, Y_te, nr['pj_mins'], nr['pj_maxs']


def evaluate(model, loader, device, pj_mins, pj_maxs):
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for x, y in loader:
            reset_ssm(model)                       # ★ reset mỗi batch
            preds.append(model(x.to(device)).cpu())
            gts.append(y)
    pred = torch.cat(preds).numpy().reshape(-1, 19, 3)
    gt   = torch.cat(gts).numpy().reshape(-1, 19, 3)
    rng = pj_maxs - pj_mins
    pred_cm = ((pred + 1) / 2 * rng + pj_mins) * 100
    gt_cm   = ((gt   + 1) / 2 * rng + pj_mins) * 100
    return {
        "mae_cm":   np.mean(np.abs(pred_cm - gt_cm)),
        "rmse_cm":  np.sqrt(np.mean((pred_cm - gt_cm) ** 2)),
        "mpjpe_cm": np.mean(np.linalg.norm(pred_cm - gt_cm, axis=-1)),
    }


def get_lr(epoch, cfg):
    if cfg.warmup_epochs > 0 and epoch < cfg.warmup_epochs:
        return cfg.lr * (epoch + 1) / cfg.warmup_epochs
    decay_start = cfg.epochs // 2
    if epoch < decay_start:
        return cfg.lr
    t = (epoch - decay_start) / max(1, cfg.epochs - decay_start)
    return cfg.lr * 0.5 * (1.0 + math.cos(math.pi * t))


def load_fp32_warmstart(model, path, device):
    if not path or not os.path.exists(path):
        print(f"⚠️  Không có {path} — train from scratch")
        return False
    ckpt = torch.load(path, map_location=device, weights_only=True)
    state = ckpt.get("model_state_dict", ckpt)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"✅ Warm-start: {path}")
    print(f"   Loaded={len(state)-len(unexpected)} Missing={len(missing)} (h_int buffer, OK)")
    clamp_weights(model)
    return True


def main():
    global FRAC_MAP
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}\n")

    cfg = MambaConfig()
    cfg.print_summary()
    FRAC_MAP = build_frac_map(cfg)

    X_tr, Y_tr, X_te, Y_te, pj_mins, pj_maxs = preprocess()
    kw = dict(num_workers=2, pin_memory=True)
    train_loader = DataLoader(MARSDataset(X_tr, Y_tr), batch_size=cfg.batch_size, shuffle=True, **kw)
    test_loader  = DataLoader(MARSDataset(X_te, Y_te), batch_size=cfg.batch_size, shuffle=False, **kw)

    model = build_model(cfg).to(device)
    model.summary()

    warm = load_fp32_warmstart(model, cfg.fp32_ckpt, device)
    out_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
    os.makedirs(out_dir, exist_ok=True)

    if warm:
        m0 = evaluate(model, test_loader, device, pj_mins, pj_maxs)
        print(f"\n📊 Baseline warm-start: MAE={m0['mae_cm']:.3f} | MPJPE={m0['mpjpe_cm']:.3f} cm")
        best_mpjpe = m0['mpjpe_cm']
        torch.save({"model_state_dict": model.state_dict()}, os.path.join(out_dir, "best_qat.pt"))
        print(f"🔒 Baseline {best_mpjpe:.3f} cm\n")
    else:
        best_mpjpe = float('inf')

    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    criterion = nn.MSELoss()

    for epoch in range(cfg.epochs):
        lr = get_lr(epoch, cfg)
        for pg in optimizer.param_groups: pg['lr'] = lr

        model.train()
        total_loss = 0.0
        t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            reset_ssm(model)                       # ★ reset mỗi batch
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
            total_loss += loss.item()

        clamp_weights(model)
        m = evaluate(model, test_loader, device, pj_mins, pj_maxs)
        print(f"Ep {epoch+1:3d}/{cfg.epochs} | lr={lr:.1e} | Loss={total_loss:.2f} | "
              f"MAE={m['mae_cm']:.3f} | RMSE={m['rmse_cm']:.3f} | MPJPE={m['mpjpe_cm']:.3f} cm | {time.time()-t0:.0f}s")

        if m['mpjpe_cm'] < best_mpjpe:
            best_mpjpe = m['mpjpe_cm']
            torch.save({"model_state_dict": model.state_dict()}, os.path.join(out_dir, "best_qat.pt"))
            print(f"  → Best! best_qat.pt (MPJPE: {best_mpjpe:.3f} cm)")

    print(f"\n✅ QAT done! Best MPJPE: {best_mpjpe:.3f} cm")
    print(f"   best_qat.pt trong dải INT8 — export thẳng, KHÔNG cần convert")


if __name__ == "__main__":
    main()

In [ ]:
# chuyển file pt từ fp32 - int8 

In [ ]:
cd /kaggle/input/code-int8

In [ ]:
import os
p = "/kaggle/input/code-int8/model"
print(os.listdir(p))
# Check mamba_model.py có BLOCK_FRAC:
with open(f"{p}/mamba_model.py") as f:
    print("BLOCK_FRAC" in f.read())   # phải True
# Check ssm.py có INT16:
with open(f"{p}/ssm.py") as f:
    print("32767" in f.read())        # phải True

In [ ]:
"""
═══════════════════════════════════════════════════════════════════════
  Chuyển best_fp32.pt → int8.pt (Per-layer FRAC)
  Mỗi layer clamp theo FRAC riêng thay vì Q5 cố định.
═══════════════════════════════════════════════════════════════════════
"""

import os, sys, math
import torch
import numpy as np

# ── Đường dẫn ──
PATH_FP32_PT   = "/kaggle/input/datasets/thanhqucl/file-pth/best_fp32 (2).pt"
PATH_INT8_CODE = "/kaggle/input/code-int8"
PATH_INT8_PT   = "/kaggle/working/int8.pt"

sys.path.insert(0, PATH_INT8_CODE)

from model.mamba_model import build_model, BLOCK_FRAC, HEAD_FRAC, PATCH_FRAC
from model.ssm import SCALE, A_SCALE, D2_BIAS_SCALE, FRAC
from train import MambaConfig

device = "cuda" if torch.cuda.is_available() else "cpu"

# ═════════════════════════════════════════════════════════════════════
# Bảng FRAC per-layer (tự build từ config)
# ═════════════════════════════════════════════════════════════════════
def build_frac_map(cfg):
    """Tạo dict: param_name → (frac, loại)"""
    frac_map = {}

    # Patch embed
    pf = PATCH_FRAC
    frac_map['embedding.proj.weight'] = (pf['w_frac'], 'weight')
    frac_map['embedding.proj.bias']   = (pf.get('bias_frac', pf['w_frac']), 'bias')

    # Mamba blocks
    for i in range(cfg.n_mamba_blocks):
        bf = BLOCK_FRAC[i]
        p = f'blocks.{i}.'

        # norm: gamma/beta luôn Q5 (RangeNorm dùng SCALE cố định)
        frac_map[p + 'norm.gamma'] = (FRAC, 'weight')
        frac_map[p + 'norm.beta']  = (FRAC, 'bias')

        # proj_x, proj_z
        frac_map[p + 'proj_x.weight'] = (bf['frac_proj_x'], 'weight')
        frac_map[p + 'proj_z.weight'] = (bf['frac_proj_z'], 'weight')

        # conv1d
        frac_map[p + 'conv1d.weight'] = (bf['frac_conv_w'], 'weight')
        frac_map[p + 'conv1d.bias']   = (bf.get('frac_conv_b', bf['frac_conv_w']), 'bias')

        # selection
        frac_map[p + 'ssm.selection.proj_b.weight']       = (bf['frac_sel_b'], 'weight')
        frac_map[p + 'ssm.selection.proj_c.weight']       = (bf['frac_sel_c'], 'weight')
        frac_map[p + 'ssm.selection.proj_delta_1.weight']  = (bf['frac_sel_d1'], 'weight')
        frac_map[p + 'ssm.selection.proj_delta_2.weight']  = (bf['frac_sel_d2'], 'weight')
        # delta2 bias: INT16 Q8, xử lý riêng
        frac_map[p + 'ssm.selection.proj_delta_2.bias']    = (8, 'delta2_bias')

        # A_log, D: giữ nguyên scheme riêng
        frac_map[p + 'ssm.A_log'] = (None, 'A_log')
        frac_map[p + 'ssm.D']     = (FRAC, 'weight')

        # out_proj
        frac_map[p + 'out_proj.weight'] = (bf['frac_outproj'], 'weight')

    # Global norm
    frac_map['norm.gamma'] = (FRAC, 'weight')
    frac_map['norm.beta']  = (FRAC, 'bias')

    # Head
    hf = HEAD_FRAC
    frac_map['head.ffn_in.weight']  = (hf['frac_ffn_in_w'], 'weight')
    frac_map['head.ffn_in.bias']    = (hf['frac_ffn_in_b'], 'bias')
    frac_map['head.ffn_out.weight'] = (hf['frac_ffn_out_w'], 'weight')
    frac_map['head.ffn_out.bias']   = (hf['frac_ffn_out_b'], 'bias')
    frac_map['head.proj.weight']    = (hf['frac_proj_w'], 'weight')
    frac_map['head.proj.bias']      = (hf['frac_proj_b'], 'bias')

    return frac_map


# ═════════════════════════════════════════════════════════════════════
# MAIN
# ═════════════════════════════════════════════════════════════════════
cfg = MambaConfig()
model = build_model(cfg).to(device)

# Load FP32 weights
ckpt = torch.load(PATH_FP32_PT, map_location=device, weights_only=True)
state = ckpt.get("model_state_dict", ckpt)

missing, unexpected = model.load_state_dict(state, strict=False)
print(f"Loaded FP32 weights: {len(state) - len(unexpected)} tensors")
if missing:
    print(f"  Missing (buffer): {missing}")

# Build FRAC map
frac_map = build_frac_map(cfg)

# Clamp per-layer
print(f"\nClamp weights → INT8 (per-layer FRAC):")
print(f"{'─'*75}")

n_clamped = 0
with torch.no_grad():
    for name, p in model.named_parameters():
        old_min, old_max = p.min().item(), p.max().item()

        if name not in frac_map:
            # Fallback: Q5
            lo, hi = -128.0 / SCALE, 127.0 / SCALE
            p.clamp_(lo, hi)
            tag = f"Q{FRAC} (fallback)"
        else:
            frac_val, kind = frac_map[name]

            if kind == 'A_log':
                limit = math.log(128.0 / float(A_SCALE))
                p.clamp_(max=limit)
                tag = f"A_log: max={limit:.4f}"

            elif kind == 'delta2_bias':
                lo = -32768.0 / float(D2_BIAS_SCALE)
                hi =  32767.0 / float(D2_BIAS_SCALE)
                p.clamp_(lo, hi)
                tag = f"INT16 Q{frac_val}"

            else:
                scale = 1 << frac_val
                lo = -128.0 / scale
                hi =  127.0 / scale
                p.clamp_(lo, hi)
                tag = f"Q{frac_val} [{lo:.4f}, {hi:.4f}]"

        new_min, new_max = p.min().item(), p.max().item()
        changed = (old_min != new_min) or (old_max != new_max)

        if changed:
            n_clamped += 1
            print(f"  ✂ {name:<45} [{old_min:+.4f}, {old_max:+.4f}] → [{new_min:+.4f}, {new_max:+.4f}]  ({tag})")
        else:
            print(f"  ✓ {name:<45} [{old_min:+.4f}, {old_max:+.4f}]  ({tag})")

print(f"\n  {n_clamped} tensors bị clamp")

# Lưu
torch.save({"model_state_dict": model.state_dict()}, PATH_INT8_PT)
print(f"\n✅ Đã lưu: {PATH_INT8_PT}")

# Verify
model2 = build_model(cfg).to(device)
ckpt2 = torch.load(PATH_INT8_PT, map_location=device, weights_only=True)
model2.load_state_dict(ckpt2["model_state_dict"])

ok = True
for name, p in model2.named_parameters():
    if name in frac_map:
        frac_val, kind = frac_map[name]
        if kind == 'A_log':
            if p.max().item() > math.log(128.0 / float(A_SCALE)) + 1e-6:
                print(f"  ⚠ {name} vượt dải A_log!")
                ok = False
        elif kind == 'delta2_bias':
            if p.min().item() < -32768/D2_BIAS_SCALE - 1e-6 or p.max().item() > 32767/D2_BIAS_SCALE + 1e-6:
                print(f"  ⚠ {name} vượt dải INT16!")
                ok = False
        elif frac_val is not None:
            scale = 1 << frac_val
            if p.min().item() < -128/scale - 1e-6 or p.max().item() > 127/scale + 1e-6:
                print(f"  ⚠ {name} vượt dải Q{frac_val}!")
                ok = False

if ok:
    print("✅ Verify OK — tất cả weights nằm trong dải per-layer")

n_params = sum(p.numel() for p in model.parameters())
print(f"\n  Params: {n_params:,}")
print(f"  FP32 file: {os.path.getsize(PATH_FP32_PT)/1024:.1f} KB")
print(f"  INT8 file: {os.path.getsize(PATH_INT8_PT)/1024:.1f} KB")

In [ ]:
# khúc này so sánh fp32 vs int8

In [ ]:
cd /kaggle/input/code-int8

In [ ]:
import sys; sys.path.insert(0, PATH_INT8_CODE)
from model.ssm import fq16   # nếu ImportError → code cũ, chưa có INT16
print("OK - code có INT16")

In [ ]:
"""
═══════════════════════════════════════════════════════════════════════
  ĐÁNH GIÁ: INT8 (clamp từ FP32) vs FP32 vs Ground Truth
  Paste vào 1 cell Kaggle, chạy → xuất comparison_results.npz
  Tải về máy → chạy demo_viewer.py để xem 3D skeleton
═══════════════════════════════════════════════════════════════════════
"""

import os, sys, time
import numpy as np
import torch

# ═════════════════════════════════════════════════════════════════════
# 1. CẤU HÌNH ĐƯỜNG DẪN
# ═════════════════════════════════════════════════════════════════════
# INT8 model code + weights (clamp từ FP32)
PATH_INT8_CODE   = "/kaggle/input/code-int8"
PATH_INT8_WEIGHT = "/kaggle/input/datasets/thanhqucl/file-pth/best_qat.pt"

# FP32 model code + weights
PATH_FP32_CODE   = "/kaggle/input/datasets/thanhqucl/code-fp-32"
PATH_FP32_WEIGHT = "/kaggle/input/datasets/thanhqucl/file-pth/best_fp32.pt"

# Data
PATH_DATA_DIR    = "/kaggle/input/datasets/thanhqucl/mars-data"
PATH_NORM_RANGE  = "/kaggle/working/data/processed/mars/norm_range.npz"

# ═════════════════════════════════════════════════════════════════════
# 2. HÀM LOAD MODEL — cách ly sys.modules tránh xung đột
# ═════════════════════════════════════════════════════════════════════
def _clear_model_modules():
    """Xóa cache module cũ để load model mới không bị lẫn"""
    for m in list(sys.modules.keys()):
        if m.startswith('model') or m == 'train':
            del sys.modules[m]


def load_int8_model(device):
    """Load INT8 RTL model (dùng code từ PATH_INT8_CODE)"""
    _clear_model_modules()
    sys.path.insert(0, PATH_INT8_CODE)

    from model.mamba_model import build_model
    from train import MambaConfig

    cfg = MambaConfig()
    model = build_model(cfg).to(device)

    ckpt = torch.load(PATH_INT8_WEIGHT, map_location=device, weights_only=True)
    model.load_state_dict(ckpt.get("model_state_dict", ckpt), strict=False)
    model.eval()

    # Lấy SCALE để fake-quantize input
    from model.ssm import SCALE
    scale = SCALE

    sys.path.remove(PATH_INT8_CODE)
    return model, scale, cfg


def load_fp32_model(device):
    """Load FP32 model (dùng code từ PATH_FP32_CODE)"""
    _clear_model_modules()
    sys.path.insert(0, PATH_FP32_CODE)

    from model.mamba_model import build_model
    from train import MambaConfig

    cfg = MambaConfig()
    model = build_model(cfg).to(device)

    ckpt = torch.load(PATH_FP32_WEIGHT, map_location=device, weights_only=True)
    model.load_state_dict(ckpt.get("model_state_dict", ckpt), strict=False)
    model.eval()

    sys.path.remove(PATH_FP32_CODE)
    return model, cfg


# ═════════════════════════════════════════════════════════════════════
# 3. CHUẨN BỊ INPUT & DENORMALIZE
# ═════════════════════════════════════════════════════════════════════
def load_norm_range(path):
    nr = np.load(path)
    return nr['feat_low'], nr['feat_high'], nr['pj_mins'], nr['pj_maxs']


def normalize_input(X_raw, feat_low, feat_high):
    """Chuẩn hóa X về [-1, 1] dùng chung cho cả 2 model"""
    X_norm = 2.0 * (X_raw - feat_low) / (feat_high - feat_low + 1e-6) - 1.0
    return np.clip(X_norm, -1.0, 1.0)


def denorm_to_cm(Y_norm, pj_mins, pj_maxs):
    """Giải chuẩn hóa output [-1,1] → cm"""
    Y_3d = Y_norm.reshape(-1, 19, 3)
    rng = pj_maxs - pj_mins
    meters = (Y_3d + 1.0) / 2.0 * rng + pj_mins
    return (meters * 100.0).astype(np.float32)


def calc_metrics(pred_cm, gt_cm):
    """Tính MAE, RMSE, MPJPE (cm)"""
    err = pred_cm - gt_cm
    mae   = np.mean(np.abs(err))
    rmse  = np.sqrt(np.mean(err ** 2))
    mpjpe = np.mean(np.linalg.norm(err, axis=-1))
    return mae, rmse, mpjpe


def per_axis_metrics(pred_cm, gt_cm):
    """Tính MAE, RMSE theo từng trục X/Y/Z"""
    results = {}
    for ax_name, ax_idx in [("X", 0), ("Y", 1), ("Z", 2)]:
        err = pred_cm[..., ax_idx] - gt_cm[..., ax_idx]
        results[ax_name] = {
            'mae':  np.mean(np.abs(err)),
            'rmse': np.sqrt(np.mean(err ** 2))
        }
    return results


# ═════════════════════════════════════════════════════════════════════
# 4. MAIN — CHẠY ĐÁNH GIÁ
# ═════════════════════════════════════════════════════════════════════
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}\n")

# Load normalization ranges
feat_low, feat_high, pj_mins, pj_maxs = load_norm_range(PATH_NORM_RANGE)

# Load test data
X_test = np.load(os.path.join(PATH_DATA_DIR, "featuremap_test.npy")).astype(np.float32)
Y_test = np.load(os.path.join(PATH_DATA_DIR, "labels_test.npy")).astype(np.float32)
print(f"Test samples: {len(X_test)}")

# Normalize input
X_norm = normalize_input(X_test, feat_low, feat_high)

# ── Load models ──
print("\nĐang load INT8 model...")
model_int8, SCALE, cfg = load_int8_model(device)

print("Đang load FP32 model...")
model_fp32, _ = load_fp32_model(device)

n_int8 = sum(p.numel() for p in model_int8.parameters())
n_fp32 = sum(p.numel() for p in model_fp32.parameters())
print(f"\nINT8 params: {n_int8:,} | FP32 params: {n_fp32:,}")

# ── Inference ──
print("\nĐang chạy inference...")
bs = 64
preds_int8, preds_fp32 = [], []

t0 = time.time()
with torch.no_grad():
    for i in range(0, len(X_norm), bs):
        xb = X_norm[i:i+bs]

        # INT8: fake quantize input (round + clamp giống RTL)
        xb_int8 = np.round(xb * SCALE) / SCALE
        x_t = torch.from_numpy(xb_int8).float().to(device)

        # Reset SSM state nếu có
        for blk in model_int8.blocks:
            if hasattr(blk.ssm, 'reset_state'):
                blk.ssm.reset_state()
        preds_int8.append(model_int8(x_t).cpu().numpy())

        # FP32: input float thuần
        x_t_fp = torch.from_numpy(xb).float().to(device)
        preds_fp32.append(model_fp32(x_t_fp).cpu().numpy())

t_inf = time.time() - t0
print(f"Inference: {t_inf:.1f}s ({len(X_norm)} samples × 2 models)")

# ── Kết quả ──
pred_int8 = np.concatenate(preds_int8)
pred_fp32 = np.concatenate(preds_fp32)

# Flatten nếu cần
if pred_int8.ndim == 3:
    pred_int8 = pred_int8.reshape(len(pred_int8), -1)
if pred_fp32.ndim == 3:
    pred_fp32 = pred_fp32.reshape(len(pred_fp32), -1)

# Denormalize → cm
Y1_cm = denorm_to_cm(pred_int8, pj_mins, pj_maxs)
Y2_cm = denorm_to_cm(pred_fp32, pj_mins, pj_maxs)

# Ground truth: reshape từ (N, 3, 19) → (N, 19, 3) rồi × 100
Y_gt_cm = (Y_test.reshape(-1, 3, 19).transpose(0, 2, 1) * 100.0).astype(np.float32)

# ── Tính metrics ──
mae1, rmse1, mpjpe1 = calc_metrics(Y1_cm, Y_gt_cm)
mae2, rmse2, mpjpe2 = calc_metrics(Y2_cm, Y_gt_cm)

ax1 = per_axis_metrics(Y1_cm, Y_gt_cm)
ax2 = per_axis_metrics(Y2_cm, Y_gt_cm)

# ── In kết quả ──
print("\n" + "=" * 75)
print("  KẾT QUẢ: INT8 (clamp từ FP32) vs FP32 vs Ground Truth")
print("=" * 75)
print(f"\n  {'Model':<28} {'MAE (cm)':>10} {'RMSE (cm)':>11} {'MPJPE (cm)':>12}")
print(f"  {'─' * 61}")
print(f"  {'INT8 (clamp từ FP32)':<28} {mae1:10.3f} {rmse1:11.3f} {mpjpe1:12.3f}")
print(f"  {'FP32 (pure float)':<28} {mae2:10.3f} {rmse2:11.3f} {mpjpe2:12.3f}")
print(f"  {'─' * 61}")
delta_mpjpe = mpjpe1 - mpjpe2
print(f"  {'Δ (INT8 − FP32)':<28} {mae1-mae2:+10.3f} {rmse1-rmse2:+11.3f} {delta_mpjpe:+12.3f}")

print(f"\n  Per-axis MAE / RMSE (cm):")
print(f"  {'─' * 61}")
for ax_name in ["X", "Y", "Z"]:
    a1, a2 = ax1[ax_name], ax2[ax_name]
    print(f"  {ax_name}  INT8: MAE={a1['mae']:.3f} RMSE={a1['rmse']:.3f}  |  "
          f"FP32: MAE={a2['mae']:.3f} RMSE={a2['rmse']:.3f}  |  "
          f"Δ MAE={a1['mae']-a2['mae']:+.3f}")

# Output diff giữa 2 model
out_diff = np.abs(Y1_cm - Y2_cm)
print(f"\n  Khoảng cách output giữa INT8 và FP32:")
print(f"    Max  = {out_diff.max():.3f} cm")
print(f"    Mean = {out_diff.mean():.3f} cm")

print(f"\n  Quantization gap: {abs(delta_mpjpe):.3f} cm", end="")
if abs(delta_mpjpe) < 1.0:
    print(" ← Tuyệt vời cho edge deployment!")
elif abs(delta_mpjpe) < 3.0:
    print(" ← Chấp nhận được")
else:
    print(" ← Lớn, cần cải thiện (tăng FRAC hoặc QAT)")

print("=" * 75)

# ── Lưu kết quả cho viewer ──
save_path = "/kaggle/working/comparison_results.npz"
np.savez(save_path, pred1_cm=Y1_cm, pred2_cm=Y2_cm, gt_cm=Y_gt_cm)
print(f"\n✅ Đã lưu: {save_path}")
print("   Tải về máy → chạy demo_viewer.py để xem 3D skeleton")

In [ ]:
# code này xuất ra mem 

In [ ]:
cd /kaggle/input/code-int8

In [ ]:
"""
Export FULL test set → single .mem files
=========================================
Output:
  all_patches.mem      — N×16×20 INT8 (cho Vivado)
  all_golden.mem       — N×57 INT8 (Python INT8 golden)
  all_int8_cm.npy      — N×19×3 float (INT8 output, cm)
  all_fp32_cm.npy      — N×19×3 float (FP32 output, cm)
  all_gt_cm.npy        — N×19×3 float (ground truth, cm)
  config.vh            — SystemVerilog header cho testbench
"""

import os, sys, shutil, time
import numpy as np
import torch

# ── Paths ──
PATH_INT8_CODE = "/kaggle/input/code-int8"
PATH_INT8_PT   = "/kaggle/input/datasets/thanhqucl/file-pth/best_qat.pt"
PATH_FP32_CODE = "/kaggle/input/datasets/thanhqucl/code-fp-32"
PATH_FP32_PT   = "/kaggle/input/datasets/thanhqucl/file-pth/best_fp32.pt"
PATH_DATA      = "/kaggle/working/data/processed/mars"
PATH_DATA_RAW  = "/kaggle/input/datasets/thanhqucl/mars-data"
PATH_NORM      = "/kaggle/working/data/processed/mars/norm_range.npz"
OUTPUT_DIR     = "/kaggle/working/rtl_full_test"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Helpers ──
def clear_modules():
    for m in list(sys.modules.keys()):
        if m.startswith('model') or m == 'train':
            del sys.modules[m]

def quant8(arr, scale):
    return np.clip(np.round(np.asarray(arr, dtype=np.float64) * scale),
                   -128, 127).astype(np.int8)

def write_flat_mem(path, data_int8):
    flat = np.asarray(data_int8).flatten().astype(np.int8).view(np.uint8)
    with open(path, 'w') as f:
        for v in flat:
            f.write(f"{v:02x}\n")
    return len(flat)

def denorm_cm(y_flat, pj_mins, pj_maxs):
    rng = pj_maxs - pj_mins
    y3d = y_flat.reshape(-1, 19, 3)
    return (((y3d + 1) / 2 * rng + pj_mins) * 100).astype(np.float32)

# ── Load data ──
nr = np.load(PATH_NORM)
feat_low, feat_high = nr['feat_low'], nr['feat_high']
pj_mins, pj_maxs = nr['pj_mins'], nr['pj_maxs']

X_raw = np.load(os.path.join(PATH_DATA_RAW, "featuremap_test.npy")).astype(np.float32)
Y_raw = np.load(os.path.join(PATH_DATA_RAW, "labels_test.npy")).astype(np.float32)
N = len(X_raw)
print(f"Full test set: {N} samples")

# Normalize input
X_norm = np.clip(2.0 * (X_raw - feat_low) / (feat_high - feat_low + 1e-6) - 1.0, -1.0, 1.0)

# Ground truth (cm)
gt_cm = (Y_raw.reshape(-1, 3, 19).transpose(0, 2, 1) * 100.0).astype(np.float32)
np.save(os.path.join(OUTPUT_DIR, "all_gt_cm.npy"), gt_cm)
print(f"  GT shape: {gt_cm.shape}")

# ═════════════════════════════════════════════════════════════════
# INT8 MODEL
# ═════════════════════════════════════════════════════════════════
print("\n── Loading INT8 model ──")
clear_modules()
sys.path.insert(0, PATH_INT8_CODE)
from model.mamba_model import build_model as build_int8
from model.ssm import SCALE as INT8_SCALE
from train import MambaConfig
cfg = MambaConfig()
model_int8 = build_int8(cfg).to("cpu")
ckpt = torch.load(PATH_INT8_PT, map_location="cpu", weights_only=True)
model_int8.load_state_dict(ckpt.get("model_state_dict", ckpt))
model_int8.eval()
SCALE = int(INT8_SCALE)
sys.path.remove(PATH_INT8_CODE)
print(f"  INT8 loaded (SCALE={SCALE})")

# ═════════════════════════════════════════════════════════════════
# FP32 MODEL
# ═════════════════════════════════════════════════════════════════
print("\n── Loading FP32 model ──")
clear_modules()
sys.path.insert(0, PATH_FP32_CODE)
from model.mamba_model import build_model as build_fp32
from train import MambaConfig as MCfg2
cfg2 = MCfg2()
model_fp32 = build_fp32(cfg2).to("cpu")
ckpt2 = torch.load(PATH_FP32_PT, map_location="cpu", weights_only=True)
model_fp32.load_state_dict(ckpt2.get("model_state_dict", ckpt2))
model_fp32.eval()
sys.path.remove(PATH_FP32_CODE)
print(f"  FP32 loaded")

# ═════════════════════════════════════════════════════════════════
# INFERENCE FULL TEST SET
# ═════════════════════════════════════════════════════════════════
print(f"\n── Inference {N} samples ──")
P_s = cfg.patch_size
bs = 1  # 1 sample at a time (bit-exact, CPU)

all_patches = []
all_golden  = []
all_int8_cm = []
all_fp32_cm = []

t0 = time.time()
with torch.no_grad():
    for i in range(N):
        x_np = X_norm[i:i+1]

        # ── INT8 path ──
        x_fq_np = np.round(x_np * SCALE) / SCALE
        x_fq = torch.from_numpy(x_fq_np).float()

        # Patches for RTL
        B, H, W, C = x_fq.shape
        xp = x_fq.reshape(B, H//P_s, P_s, W//P_s, P_s, C)
        xp = xp.permute(0, 1, 3, 5, 2, 4).reshape(B, cfg.seq_len, C, P_s, P_s)
        all_patches.append(quant8(xp.numpy()[0], SCALE))

        for blk in model_int8.blocks:
            blk.ssm.reset_state()
        y_int8 = model_int8(x_fq).detach().cpu().numpy()
        all_golden.append(quant8(y_int8, SCALE))
        all_int8_cm.append(denorm_cm(y_int8, pj_mins, pj_maxs))

        # ── FP32 path ──
        x_fp = torch.from_numpy(x_np).float()
        y_fp32 = model_fp32(x_fp).detach().cpu().numpy()
        all_fp32_cm.append(denorm_cm(y_fp32, pj_mins, pj_maxs))

        if (i + 1) % 500 == 0 or i == 0:
            print(f"  {i+1}/{N} ({time.time()-t0:.0f}s)")

elapsed = time.time() - t0
print(f"  Done in {elapsed:.0f}s")

# ═════════════════════════════════════════════════════════════════
# GHI FILES
# ═════════════════════════════════════════════════════════════════
print(f"\n── Writing files ──")

# .mem cho Vivado
patches_flat = np.concatenate([p.flatten() for p in all_patches])
n_in = write_flat_mem(os.path.join(OUTPUT_DIR, "all_patches.mem"), patches_flat)
print(f"  all_patches.mem: {n_in:,} bytes ({N}×{cfg.seq_len}×20)")

golden_flat = np.concatenate([g.flatten() for g in all_golden])
n_out = write_flat_mem(os.path.join(OUTPUT_DIR, "all_golden.mem"), golden_flat)
print(f"  all_golden.mem:  {n_out:,} bytes ({N}×57)")

# .npy cho Python post-processing
int8_cm = np.concatenate(all_int8_cm, axis=0)
fp32_cm = np.concatenate(all_fp32_cm, axis=0)
np.save(os.path.join(OUTPUT_DIR, "all_int8_cm.npy"), int8_cm)
np.save(os.path.join(OUTPUT_DIR, "all_fp32_cm.npy"), fp32_cm)
np.savez(os.path.join(OUTPUT_DIR, "norm_range.npz"),
         pj_mins=pj_mins, pj_maxs=pj_maxs)
print(f"  all_int8_cm.npy: {int8_cm.shape}")
print(f"  all_fp32_cm.npy: {fp32_cm.shape}")

# SystemVerilog config header
vh_path = os.path.join(OUTPUT_DIR, "batch_config.vh")
with open(vh_path, 'w') as f:
    f.write(f"localparam int NUM_SAMPLES = {N};\n")
print(f"  batch_config.vh: NUM_SAMPLES={N}")

# ── Metrics ──
def metrics(pred, gt):
    mae = np.mean(np.abs(pred - gt))
    rmse = np.sqrt(np.mean((pred - gt)**2))
    mpjpe = np.mean(np.linalg.norm(pred - gt, axis=-1))
    return mae, rmse, mpjpe

mi8  = metrics(int8_cm, gt_cm)
mfp  = metrics(fp32_cm, gt_cm)

print(f"\n{'='*60}")
print(f"  METRICS trên {N} samples")
print(f"{'='*60}")
print(f"  {'Model':<16} {'MAE':>8} {'RMSE':>8} {'MPJPE':>8}")
print(f"  {'-'*40}")
print(f"  {'INT8 (=RTL)':<16} {mi8[0]:8.3f} {mi8[1]:8.3f} {mi8[2]:8.3f}")
print(f"  {'FP32':<16} {mfp[0]:8.3f} {mfp[1]:8.3f} {mfp[2]:8.3f}")
print(f"  {'Δ (INT8-FP32)':<16} {mi8[0]-mfp[0]:+8.3f} {mi8[1]-mfp[1]:+8.3f} {mi8[2]-mfp[2]:+8.3f}")
print(f"{'='*60}")

# ── ZIP ──
zip_path = shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print(f"\n✅ {zip_path}")

In [ ]:
"""
export.py — Export Full eMamba weights + I/O sang .mem
=======================================================
Per-layer FRAC + INT16 SSM path.
Bit-exact 1-1 với emamba_top.sv.

CẤU TRÚC FILE .mem:
  Patch embed:     patch_embed_w.mem, patch_embed_b.mem
  Block i norm:    b{i}_norm_w.mem, b{i}_norm_b.mem
  Block i proj:    b{i}_proj_x_w.mem, b{i}_proj_z_w.mem
  Block i conv1d:  b{i}_conv1d_w.mem, b{i}_conv1d_b.mem
  Block i sel:     b{i}_proj_b_w.mem, b{i}_proj_c_w.mem
                   b{i}_proj_delta1_w.mem, b{i}_proj_delta2_w.mem
                   b{i}_proj_delta2_b.mem (INT16 Q8)
  Block i ssm:     b{i}_A.mem, b{i}_D.mem
  Block i out:     b{i}_out_proj_w.mem
  Global norm:     norm_w.mem, norm_b.mem
  Head:            head_ffn_in_w/b.mem, head_ffn_out_w/b.mem, head_proj_w/b.mem

  I/O golden:
    INT8  (.mem)  — patches, embed, norm, proj, conv, sel, block, pool, head, logits
    INT16 (.mem)  — SSM output, gating output (INT16 Q5 path)
"""

import os, sys
import numpy as np
import torch
import shutil

# Xóa cache module cũ
for m in list(sys.modules.keys()):
    if m.startswith('model') or m == 'train':
        del sys.modules[m]

for p in [".", "/kaggle/working"]:
    if p not in sys.path:
        sys.path.insert(0, p)

from model.mamba_model import build_model, BLOCK_FRAC, HEAD_FRAC, PATCH_FRAC
from model.ssm import (FRAC, SCALE, A_FRAC, A_SCALE,
                        D2_BIAS_FRAC, D2_BIAS_SCALE, rshift, fq, rtl_relu)
from train import MambaConfig

# ── Config ──
PTH_PATH = "/kaggle/input/datasets/thanhqucl/file-pth/best_qat.pt"
BASE_DIR  = "/kaggle/working/emamba_export_full"
DIR_WB    = os.path.join(BASE_DIR, "weight_bias")
DIR_IO    = os.path.join(BASE_DIR, "input_output")
os.makedirs(DIR_WB, exist_ok=True)
os.makedirs(DIR_IO, exist_ok=True)

# ── Load model ──
cfg   = MambaConfig()
model = build_model(cfg)
ckpt = torch.load(PTH_PATH, map_location="cpu", weights_only=True)
state = ckpt.get("model_state_dict", ckpt)
model.load_state_dict(state, strict=False)
model.eval()
print(f"[OK] Loaded: {PTH_PATH}\n")
model.summary()

# ── Utilities ──
def quant8(arr, scale=SCALE):
    return np.clip(np.round(np.asarray(arr, dtype=np.float64) * scale),
                   -128, 127).astype(np.int8)

def quant16(arr, scale=SCALE):
    return np.clip(np.round(np.asarray(arr, dtype=np.float64) * scale),
                   -32768, 32767).astype(np.int16)

def write_int8(path, arr):
    flat = np.asarray(arr).flatten().astype(np.int8).view(np.uint8)
    with open(path, "w") as f:
        for v in flat:
            f.write(f"{v:02x}\n")

def write_int16_le(path, arr):
    flat = np.asarray(arr).flatten().astype(np.int16).view(np.uint16)
    with open(path, "w") as f:
        for v in flat:
            f.write(f"{v:04x}\n")

def wb8(fname, arr_float, scale=SCALE):
    arr_np = np.asarray(arr_float, dtype=np.float32)
    q = quant8(arr_np, scale)
    write_int8(os.path.join(DIR_WB, fname), q)
    print(f"  [W/B] {fname:<32} shape={str(arr_np.shape):<20} max_int={int(np.abs(q).max())}/127")

def wb_A(fname, A_log_float):
    A_real = -np.exp(np.asarray(A_log_float, dtype=np.float64))
    q = quant8(A_real, A_SCALE)
    write_int8(os.path.join(DIR_WB, fname), q)
    print(f"  [W/B] {fname:<32} shape={str(A_real.shape):<20} max_int={int(np.abs(q).max())}/127 [A_SCALE={A_SCALE}]")

def wb_d2bias(fname, bias_float):
    q = quant16(bias_float, D2_BIAS_SCALE)
    write_int16_le(os.path.join(DIR_WB, fname), q)
    print(f"  [W/B] {fname:<32} shape={str(q.shape):<20} INT16 Q{D2_BIAS_FRAC}")

def io8(name, tensor_or_array):
    arr = (tensor_or_array.detach().cpu().numpy()
           if isinstance(tensor_or_array, torch.Tensor)
           else np.asarray(tensor_or_array))
    write_int8(os.path.join(DIR_IO, f"{name}.mem"), quant8(arr, SCALE))
    print(f"  [I/O] {(name+'.mem'):<38} shape={str(arr.shape)}")

def io16(name, tensor_or_array):
    """INT16 Q5 golden — cho SSM/gating output"""
    arr = (tensor_or_array.detach().cpu().numpy()
           if isinstance(tensor_or_array, torch.Tensor)
           else np.asarray(tensor_or_array))
    q = quant16(arr, SCALE)
    write_int16_le(os.path.join(DIR_IO, f"{name}.mem"), q)
    print(f"  [I/O] {(name+'.mem'):<38} shape={str(arr.shape)} [INT16]")

def get(key):
    return model.state_dict()[key].detach().cpu().numpy()


# ══════════════════════════════════════════════════════════════════════
# PHẦN 1: WEIGHT & BIAS
# ══════════════════════════════════════════════════════════════════════
print("=" * 70)
print("PHẦN 1: WEIGHT & BIAS")
print("=" * 70)

print("\n--- Patch Embedding (per-layer FRAC) ---")
wb8("patch_embed_w.mem", get("embedding.proj.weight"), 1 << PATCH_FRAC['w_frac'])
wb8("patch_embed_b.mem", get("embedding.proj.bias"),   1 << PATCH_FRAC.get('bias_frac', PATCH_FRAC['w_frac']))

print("\n--- Mamba Blocks (per-layer FRAC) ---")
for i in range(cfg.n_mamba_blocks):
    p = f"blocks.{i}."
    pfx = f"b{i}_"
    bf = BLOCK_FRAC[i]
    print(f"\n Block {i}: proj_x=Q{bf['frac_proj_x']} proj_z=Q{bf['frac_proj_z']} "
          f"conv=Q{bf['frac_conv_w']}/b{bf.get('frac_conv_b', bf['frac_conv_w'])} "
          f"out=Q{bf['frac_outproj']}")
    # norm: luôn Q5 (RangeNorm dùng SCALE cố định)
    wb8(f"{pfx}norm_w.mem",        get(p + "norm.gamma"),                    1 << FRAC)
    wb8(f"{pfx}norm_b.mem",        get(p + "norm.beta"),                     1 << FRAC)
    # proj_x, proj_z: per-layer FRAC
    wb8(f"{pfx}proj_x_w.mem",     get(p + "proj_x.weight"),                  1 << bf['frac_proj_x'])
    wb8(f"{pfx}proj_z_w.mem",     get(p + "proj_z.weight"),                  1 << bf['frac_proj_z'])
    # conv1d: weight + bias FRAC riêng
    wb8(f"{pfx}conv1d_w.mem",      get(p + "conv1d.weight"),                 1 << bf['frac_conv_w'])
    wb8(f"{pfx}conv1d_b.mem",      get(p + "conv1d.bias"),                   1 << bf.get('frac_conv_b', bf['frac_conv_w']))
    # selection
    wb8(f"{pfx}proj_b_w.mem",      get(p + "ssm.selection.proj_b.weight"),   1 << bf['frac_sel_b'])
    wb8(f"{pfx}proj_c_w.mem",      get(p + "ssm.selection.proj_c.weight"),   1 << bf['frac_sel_c'])
    wb8(f"{pfx}proj_delta1_w.mem", get(p + "ssm.selection.proj_delta_1.weight"), 1 << bf['frac_sel_d1'])
    wb8(f"{pfx}proj_delta2_w.mem", get(p + "ssm.selection.proj_delta_2.weight"), 1 << bf['frac_sel_d2'])
    wb_d2bias(f"{pfx}proj_delta2_b.mem",
              get(p + "ssm.selection.proj_delta_2.bias"))
    # A, D: scheme riêng
    wb_A(f"{pfx}A.mem",            get(p + "ssm.A_log"))
    wb8(f"{pfx}D.mem",             get(p + "ssm.D"),                         1 << FRAC)
    # out_proj
    wb8(f"{pfx}out_proj_w.mem",    get(p + "out_proj.weight"),               1 << bf['frac_outproj'])

print("\n--- Global RangeNorm (Q5) ---")
wb8("norm_w.mem", get("norm.gamma"), 1 << FRAC)
wb8("norm_b.mem", get("norm.beta"),  1 << FRAC)

print("\n--- Residual Head (per-layer FRAC) ---")
hf = HEAD_FRAC
wb8("head_ffn_in_w.mem",  get("head.ffn_in.weight"),  1 << hf['frac_ffn_in_w'])
wb8("head_ffn_in_b.mem",  get("head.ffn_in.bias"),    1 << hf['frac_ffn_in_b'])
wb8("head_ffn_out_w.mem", get("head.ffn_out.weight"), 1 << hf['frac_ffn_out_w'])
wb8("head_ffn_out_b.mem", get("head.ffn_out.bias"),   1 << hf['frac_ffn_out_b'])
wb8("head_proj_w.mem",    get("head.proj.weight"),    1 << hf['frac_proj_w'])
wb8("head_proj_b.mem",    get("head.proj.bias"),      1 << hf['frac_proj_b'])


# ══════════════════════════════════════════════════════════════════════
# PHẦN 2: I/O — HOOK-BASED (bit-exact 100%)
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PHẦN 2: I/O (HOOK-BASED — BIT-EXACT)")
print("=" * 70)

from model.ssm import INV_ROM, to_long

def hp_pool(tokens_f, seq_len):
    inv_seq = 32768 // seq_len
    x_i = (tokens_f.double() * SCALE).round().long().clamp(-128, 127)
    acc = x_i.sum(dim=1)
    pool = ((acc * inv_seq + (1 << 14)) >> 15).clamp(-128, 127)
    return pool.float() / SCALE

# ── Input ──
torch.manual_seed(42)
x_raw = torch.randn(1, cfg.input_height, cfg.input_width, cfg.in_channels)
x_raw = x_raw.clamp(-127.0 / SCALE, 127.0 / SCALE)
x_fq = (x_raw * SCALE).round().clamp(-128, 127) / SCALE
io8("00_input_raw", x_fq)

P_s = cfg.patch_size
B, H, W, C = x_fq.shape
xp = x_fq.reshape(B, H//P_s, P_s, W//P_s, P_s, C)
xp = xp.permute(0, 1, 3, 5, 2, 4).reshape(B, cfg.seq_len, C, P_s, P_s)
xp_int = (xp * SCALE).round().clamp(-128, 127).numpy().astype(np.int8)
for pi in range(cfg.seq_len):
    write_int8(os.path.join(DIR_IO, f"patch_{pi:02d}.mem"), xp_int[0, pi])
print(f"  [I/O] {cfg.seq_len} patches → patch_XX.mem  ({C}×{P_s}×{P_s} bytes)")

with torch.no_grad():
    # ══════════════════════════════════════════════════════════════
    # HOOK-BASED: 1 forward, bắt tất cả intermediates
    # ══════════════════════════════════════════════════════════════
    cap = {}
    hooks = []

    def out_hook(name):
        def fn(m, inp, out):
            cap[name] = out.detach().clone()
        return fn

    sel_data = {}
    for bi in range(cfg.n_mamba_blocks):
        sel_data[bi] = {'B': [], 'C': [], 'delta': []}

    def make_sel_hook(bi):
        def fn(m, inp, out):
            B_t, C_t, delta_t = out
            sel_data[bi]['B'].append(B_t.detach().clone())
            sel_data[bi]['C'].append(C_t.detach().clone())
            sel_data[bi]['delta'].append(delta_t.detach().clone())
        return fn

    # Register hooks
    hooks.append(model.embedding.register_forward_hook(out_hook('embed')))
    for bi, blk in enumerate(model.blocks):
        hooks.append(blk.norm.register_forward_hook(out_hook(f'b{bi}_norm')))
        hooks.append(blk.proj_x.register_forward_hook(out_hook(f'b{bi}_proj_x')))
        hooks.append(blk.proj_z.register_forward_hook(out_hook(f'b{bi}_proj_z')))
        hooks.append(blk.silu.register_forward_hook(out_hook(f'b{bi}_silu')))
        hooks.append(blk.conv1d.register_forward_hook(out_hook(f'b{bi}_conv1d')))
        hooks.append(blk.ssm.selection.register_forward_hook(make_sel_hook(bi)))
        hooks.append(blk.ssm.register_forward_hook(out_hook(f'b{bi}_ssm')))
        hooks.append(blk.out_proj.register_forward_hook(out_hook(f'b{bi}_out_proj')))
        hooks.append(blk.register_forward_hook(out_hook(f'b{bi}_block')))
    hooks.append(model.norm.register_forward_hook(out_hook('global_norm')))
    hooks.append(model.head.ffn_in.register_forward_hook(out_hook('head_ffn_in')))
    hooks.append(model.head.ffn_out.register_forward_hook(out_hook('head_ffn_out')))
    hooks.append(model.head.proj.register_forward_hook(out_hook('head_proj')))

    # Reset + forward
    for blk in model.blocks:
        blk.ssm.reset_state()
    logits = model(x_fq)

    for h in hooks:
        h.remove()

    # ── Dump intermediates ──
    io8("01_patch_embed_out", cap['embed'])

    for bi in range(cfg.n_mamba_blocks):
        pfx = f"b{bi}"

        # INT8 layers
        io8(f"{pfx}_01_norm_out", cap[f'{pfx}_norm'])

        x1 = cap[f'{pfx}_proj_x']
        z  = cap[f'{pfx}_proj_z']
        io8(f"{pfx}_02_x1", x1)
        io8(f"{pfx}_02_z",  z)

        io8(f"{pfx}_03_silu_out", cap[f'{pfx}_silu'])
        io8(f"{pfx}_04_conv1d_out", cap[f'{pfx}_conv1d'])

        io8(f"{pfx}_05_B",     torch.stack(sel_data[bi]['B'],     dim=1))
        io8(f"{pfx}_05_C",     torch.stack(sel_data[bi]['C'],     dim=1))
        io8(f"{pfx}_05_delta", torch.stack(sel_data[bi]['delta'], dim=1))

        # ★ INT16 layers (SSM path)
        io16(f"{pfx}_06_ssm_out", cap[f'{pfx}_ssm'])

        # Gating golden (INT16)
        y_ssm = cap[f'{pfx}_ssm']
        z_act = cap[f'{pfx}_silu']
        y_i = (y_ssm * SCALE).round().clamp(-32768, 32767)    # INT16
        z_i = (z_act * SCALE).round().clamp(-128, 127)        # INT8
        mult = y_i * z_i
        gated = ((mult + (1 << (FRAC-1))).div(SCALE, rounding_mode='floor')
                 .clamp(-32768, 32767)).float() / SCALE
        io16(f"{pfx}_07_gated_out", gated)

        # out_proj output (INT8 — requantize tại đây)
        io8(f"{pfx}_08_out_proj_out", cap[f'{pfx}_out_proj'])

        # Block output (INT8)
        io8(f"{pfx}_09_block_out", cap[f'{pfx}_block'])

    # Pool (INT8)
    tokens_final = cap[f'b{cfg.n_mamba_blocks-1}_block']
    pool_out = hp_pool(tokens_final, cfg.seq_len)
    io8("10_mean_pool_out", pool_out)
    d = (model.pool(tokens_final) - pool_out).abs().max().item()
    print(f"  [CHECK] pool  : diff={d:.6f} {'✓ bit-exact' if d == 0 else '✗ MISMATCH'}")

    # Global norm (INT8)
    norm_out = cap['global_norm']
    io8("11_global_norm_out", norm_out)

    # Head (INT8)
    x_res_i = (norm_out * SCALE).round().clamp(-128, 127)
    io8("12_head_xres", x_res_i / SCALE)
    io8("13_head_ffn_in_out", cap['head_ffn_in'])
    h_relu = fq(rtl_relu(cap['head_ffn_in']))
    io8("14_head_relu_out", h_relu)
    io8("15_head_ffn_out_out", cap['head_ffn_out'])
    h_out_i = (cap['head_ffn_out'] * SCALE).round().clamp(-128, 127)
    x_add = (x_res_i + h_out_i).clamp(-128, 127) / SCALE
    io8("16_head_residual_out", x_add)

    # Logits (INT8) — trực tiếp từ model
    io8("17_logits_out", logits)
    print(f"  [CHECK] logits: ✓ bit-exact (từ model(x_fq) trực tiếp)")
    print(f"  [RESULT] logits range: [{logits.min().item():.4f}, {logits.max().item():.4f}]")


# ══════════════════════════════════════════════════════════════════════
# PHẦN 3: ZIP
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
zip_path = "/kaggle/working/emamba_export_rtl_full"
shutil.make_archive(zip_path, 'zip', BASE_DIR)
print(f"✅ Xong! File: {zip_path}.zip")
print(f"   Weight/Bias : {DIR_WB}")
print(f"   I/O golden  : {DIR_IO}")

In [ ]:
# code này tính frac từng layer

In [ ]:
cd /kaggle/input/datasets/thanhqucl/code-int8

In [ ]:
# ── Tự tính FRAC tối ưu per-layer ──
print("\n" + "=" * 70)
print("RECOMMENDED PER-LAYER FRAC (từ max_int thực tế)")
print("=" * 70)
import math
for name, val in model.state_dict().items():
    arr = val.detach().cpu().numpy()
    if 'A_log' in name:
        real_max = float(np.abs(-np.exp(arr)).max())
        scale = A_SCALE
    elif 'proj_delta_2.bias' in name:
        real_max = float(np.abs(arr).max())
        scale = D2_BIAS_SCALE
        q = np.clip(np.round(arr * scale), -32768, 32767)
        print(f"  {name:<45} max={real_max:.4f}  INT16 Q8 (giữ nguyên)")
        continue
    else:
        real_max = float(np.abs(arr).max())
        scale = SCALE
    
    if real_max < 1e-9:
        print(f"  {name:<45} max≈0 → FRAC=5 (default)")
        continue
    
    best_frac = 5
    for f in range(5, 12):
        new_max_int = round(real_max * (2**f))
        if new_max_int <= 127:
            best_frac = f
        else:
            break
    
    cur_int = round(real_max * scale)
    new_int = round(real_max * (2**best_frac))
    gain = best_frac - 5
    marker = " ★★" if gain >= 2 else (" ★" if gain >= 1 else "")
    print(f"  {name:<45} max={real_max:.4f}  Q5:{cur_int:>3}/127 → Q{best_frac}:{new_int:>3}/127  +{gain}{marker}")

In [ ]:
"""
═══════════════════════════════════════════════════════════════════════
  So sánh FP32 vs INT8 per-layer — độ lệch từng layer khi xử lý 1 sample
═══════════════════════════════════════════════════════════════════════
Paste vào 1 cell Kaggle, chạy.
Output: bảng MAE/MaxErr/Correlation từng layer + tích lũy error.
"""

import os, sys, time
import numpy as np
import torch

# ═════════════════════════════════════════════════════════════════════
# CONFIG
# ═════════════════════════════════════════════════════════════════════
PATH_INT8_CODE = "/kaggle/working"
PATH_INT8_PT   = "/kaggle/input/datasets/thanhqucl/file-pth/best_qat.pt"
PATH_FP32_CODE = "/kaggle/input/datasets/thanhqucl/code-fp-32"
PATH_FP32_PT   = "/kaggle/input/datasets/thanhqucl/file-pth/best_fp32.pt"
PATH_DATA      = "/kaggle/working/data/processed/mars"

SAMPLE_IDX = 0  # Sample nào để so sánh (0 = đầu tiên)

# ═════════════════════════════════════════════════════════════════════
# HELPERS
# ═════════════════════════════════════════════════════════════════════
def clear_modules():
    for m in list(sys.modules.keys()):
        if m.startswith('model') or m == 'train':
            del sys.modules[m]

def capture_all(model, x_input, model_name, n_blocks=2):
    """Chạy model 1 lần, hooks bắt tất cả intermediates."""
    cap = {}
    hooks = []
    sel_data = {}

    def out_hook(name):
        def fn(m, inp, out):
            cap[name] = out.detach().clone()
        return fn

    def in_hook(name):
        def fn(m, inp):
            if len(inp) > 0:
                cap[name + '_input'] = inp[0].detach().clone()
        return fn

    for bi in range(n_blocks):
        sel_data[bi] = {'B': [], 'C': [], 'delta': []}

    def make_sel_hook(bi):
        def fn(m, inp, out):
            if isinstance(out, tuple) and len(out) == 3:
                B_t, C_t, delta_t = out
                sel_data[bi]['B'].append(B_t.detach().clone())
                sel_data[bi]['C'].append(C_t.detach().clone())
                sel_data[bi]['delta'].append(delta_t.detach().clone())
        return fn

    # Register hooks
    hooks.append(model.embedding.register_forward_hook(out_hook('embed')))

    for bi, blk in enumerate(model.blocks):
        hooks.append(blk.norm.register_forward_hook(out_hook(f'b{bi}_norm')))

        # proj_x + proj_z (INT8) hoặc proj_x + proj_z (FP32 mới)
        if hasattr(blk, 'proj_x'):
            hooks.append(blk.proj_x.register_forward_hook(out_hook(f'b{bi}_proj_x')))
            hooks.append(blk.proj_z.register_forward_hook(out_hook(f'b{bi}_proj_z')))
        elif hasattr(blk, 'in_proj'):
            hooks.append(blk.in_proj.register_forward_hook(out_hook(f'b{bi}_in_proj')))

        hooks.append(blk.silu.register_forward_hook(out_hook(f'b{bi}_silu')))
        hooks.append(blk.conv1d.register_forward_hook(out_hook(f'b{bi}_conv1d')))

        if hasattr(blk.ssm, 'selection'):
            hooks.append(blk.ssm.selection.register_forward_hook(make_sel_hook(bi)))

        hooks.append(blk.ssm.register_forward_hook(out_hook(f'b{bi}_ssm')))
        hooks.append(blk.out_proj.register_forward_hook(out_hook(f'b{bi}_out_proj')))
        hooks.append(blk.register_forward_hook(out_hook(f'b{bi}_block')))

    hooks.append(model.pool.register_forward_hook(out_hook('pool')))
    hooks.append(model.norm.register_forward_hook(out_hook('global_norm')))

    if hasattr(model.head, 'ffn_in'):
        hooks.append(model.head.ffn_in.register_forward_hook(out_hook('head_ffn_in')))
        hooks.append(model.head.ffn_out.register_forward_hook(out_hook('head_ffn_out')))
    hooks.append(model.head.proj.register_forward_hook(out_hook('head_proj')))
    hooks.append(model.head.register_forward_hook(out_hook('head')))

    # Forward
    with torch.no_grad():
        if hasattr(model, 'blocks'):
            for blk in model.blocks:
                if hasattr(blk.ssm, 'reset_state'):
                    blk.ssm.reset_state()
        output = model(x_input)

    for h in hooks:
        h.remove()

    # Merge selection data
    for bi in range(n_blocks):
        if sel_data[bi]['B']:
            cap[f'b{bi}_sel_B']     = torch.stack(sel_data[bi]['B'], dim=1)
            cap[f'b{bi}_sel_C']     = torch.stack(sel_data[bi]['C'], dim=1)
            cap[f'b{bi}_sel_delta'] = torch.stack(sel_data[bi]['delta'], dim=1)

    # Split in_proj nếu là model cũ
    for bi in range(n_blocks):
        if f'b{bi}_in_proj' in cap and f'b{bi}_proj_x' not in cap:
            d_inner = cap[f'b{bi}_in_proj'].shape[-1] // 2
            cap[f'b{bi}_proj_x'] = cap[f'b{bi}_in_proj'][..., :d_inner]
            cap[f'b{bi}_proj_z'] = cap[f'b{bi}_in_proj'][..., d_inner:]

    cap['output'] = output.detach().clone()
    return cap


def compare_tensors(a, b):
    """So sánh 2 tensor, trả về dict metrics."""
    a_np = a.cpu().numpy().astype(np.float64).flatten()
    b_np = b.cpu().numpy().astype(np.float64).flatten()

    diff = np.abs(a_np - b_np)
    mae     = np.mean(diff)
    max_err = np.max(diff)
    rmse    = np.sqrt(np.mean(diff**2))

    # Correlation
    if np.std(a_np) > 1e-10 and np.std(b_np) > 1e-10:
        corr = np.corrcoef(a_np, b_np)[0, 1]
    else:
        corr = 1.0 if mae < 1e-10 else 0.0

    # Range
    range_fp32 = float(np.max(np.abs(a_np))) if len(a_np) > 0 else 0
    range_int8 = float(np.max(np.abs(b_np))) if len(b_np) > 0 else 0

    return dict(mae=mae, max_err=max_err, rmse=rmse, corr=corr,
                range_fp32=range_fp32, range_int8=range_int8, n=len(a_np))


# ═════════════════════════════════════════════════════════════════════
# LOAD DATA
# ═════════════════════════════════════════════════════════════════════
test_data = np.load(f"{PATH_DATA}/test.npz")
X_test = test_data['X'].astype(np.float32)
x_sample = torch.from_numpy(X_test[SAMPLE_IDX:SAMPLE_IDX+1]).float()
print(f"Sample {SAMPLE_IDX}: shape={x_sample.shape}")

# ═════════════════════════════════════════════════════════════════════
# LOAD FP32 MODEL
# ═════════════════════════════════════════════════════════════════════
print("\n── Loading FP32 model ──")
clear_modules()
sys.path.insert(0, PATH_FP32_CODE)
from model.mamba_model import build_model as build_fp32
from train import MambaConfig as MCfg_fp32
cfg_fp32 = MCfg_fp32()
model_fp32 = build_fp32(cfg_fp32).cpu()
ckpt_fp32 = torch.load(PATH_FP32_PT, map_location="cpu", weights_only=True)
model_fp32.load_state_dict(ckpt_fp32.get("model_state_dict", ckpt_fp32), strict=False)
model_fp32.eval()
if PATH_FP32_CODE in sys.path:
    sys.path.remove(PATH_FP32_CODE)
print("  FP32 loaded")

# ═════════════════════════════════════════════════════════════════════
# LOAD INT8 MODEL
# ═════════════════════════════════════════════════════════════════════
print("\n── Loading INT8 model ──")
clear_modules()
sys.path.insert(0, PATH_INT8_CODE)
from model.mamba_model import build_model as build_int8
from model.ssm import SCALE
from train import MambaConfig as MCfg_int8
cfg_int8 = MCfg_int8()
model_int8 = build_int8(cfg_int8).cpu()
ckpt_int8 = torch.load(PATH_INT8_PT, map_location="cpu", weights_only=True)
model_int8.load_state_dict(ckpt_int8.get("model_state_dict", ckpt_int8), strict=False)
model_int8.eval()
print("  INT8 loaded")

# ═════════════════════════════════════════════════════════════════════
# CAPTURE INTERMEDIATES
# ═════════════════════════════════════════════════════════════════════
print("\n── Capturing FP32 intermediates ──")
cap_fp32 = capture_all(model_fp32, x_sample, "FP32", n_blocks=cfg_fp32.n_mamba_blocks)

# INT8: fake quantize input
x_int8 = (x_sample * SCALE).round().clamp(-128, 127) / SCALE
print("── Capturing INT8 intermediates ──")
cap_int8 = capture_all(model_int8, x_int8, "INT8", n_blocks=cfg_int8.n_mamba_blocks)

# ═════════════════════════════════════════════════════════════════════
# COMPARE PER-LAYER
# ═════════════════════════════════════════════════════════════════════
# Danh sách layers theo pipeline order
LAYER_ORDER = [
    ('input',       'Input (fake quant)'),
    ('embed',       'Patch Embed (Conv2D)'),
]

for bi in range(cfg_int8.n_mamba_blocks):
    LAYER_ORDER += [
        (f'b{bi}_norm',      f'Block {bi} — Norm'),
        (f'b{bi}_proj_x',    f'Block {bi} — proj_x (lower)'),
        (f'b{bi}_proj_z',    f'Block {bi} — proj_z (upper)'),
        (f'b{bi}_silu',      f'Block {bi} — SiLU(z)'),
        (f'b{bi}_conv1d',    f'Block {bi} — Conv1D(x1)'),
        (f'b{bi}_sel_B',     f'Block {bi} — Selection B'),
        (f'b{bi}_sel_C',     f'Block {bi} — Selection C'),
        (f'b{bi}_sel_delta', f'Block {bi} — Selection delta'),
        (f'b{bi}_ssm',       f'Block {bi} — SSM output'),
        (f'b{bi}_out_proj',  f'Block {bi} — out_proj'),
        (f'b{bi}_block',     f'Block {bi} — Block output'),
    ]

LAYER_ORDER += [
    ('pool',         'Mean Pool'),
    ('global_norm',  'Global Norm'),
    ('head_ffn_in',  'Head — ffn_in'),
    ('head_ffn_out', 'Head — ffn_out'),
    ('head_proj',    'Head — proj'),
    ('output',       'Final Output (logits)'),
]

# Thêm input vào captures
cap_fp32['input'] = x_sample
cap_int8['input'] = x_int8

print(f"\n{'='*90}")
print(f"  PER-LAYER COMPARISON: FP32 vs INT8 — Sample {SAMPLE_IDX}")
print(f"{'='*90}")
print(f"  {'Layer':<32} {'MAE':>8} {'MaxErr':>8} {'RMSE':>8} {'Corr':>7} {'|FP32|':>7} {'|INT8|':>7} {'N':>6}")
print(f"  {'─'*86}")

prev_mae = 0
results = []
for key, label in LAYER_ORDER:
    if key not in cap_fp32 or key not in cap_int8:
        continue

    m = compare_tensors(cap_fp32[key], cap_int8[key])
    results.append((key, label, m))

    # Error growth indicator
    if m['mae'] > prev_mae * 1.5 and prev_mae > 0.001:
        marker = " ◄◄"
    elif m['mae'] > prev_mae and prev_mae > 0.001:
        marker = " ◄"
    else:
        marker = ""

    print(f"  {label:<32} {m['mae']:8.4f} {m['max_err']:8.4f} {m['rmse']:8.4f} "
          f"{m['corr']:7.4f} {m['range_fp32']:7.3f} {m['range_int8']:7.3f} "
          f"{m['n']:6d}{marker}")

    prev_mae = m['mae']

# ═════════════════════════════════════════════════════════════════════
# SUMMARY
# ═════════════════════════════════════════════════════════════════════
print(f"\n{'='*90}")
print(f"  SUMMARY")
print(f"{'='*90}")

# Top 5 worst layers
sorted_by_mae = sorted(results, key=lambda x: x[2]['mae'], reverse=True)
print(f"\n  Top 5 layers có MAE cao nhất (bottleneck quantization):")
for i, (key, label, m) in enumerate(sorted_by_mae[:5]):
    print(f"    {i+1}. {label:<30} MAE={m['mae']:.4f}  MaxErr={m['max_err']:.4f}  Corr={m['corr']:.4f}")

# Error growth chain
print(f"\n  Error growth qua pipeline:")
for key, label, m in results:
    bar_len = min(int(m['mae'] * 100), 50)
    bar = '█' * bar_len + '░' * (50 - bar_len)
    print(f"    {label:<28} {bar} {m['mae']:.4f}")

# Correlation < 0.99
print(f"\n  Layers có correlation < 0.99 (lệch nhiều nhất):")
for key, label, m in results:
    if m['corr'] < 0.99:
        print(f"    {label:<30} Corr={m['corr']:.4f}")

# Final output comparison
if 'output' in cap_fp32 and 'output' in cap_int8:
    m_out = compare_tensors(cap_fp32['output'], cap_int8['output'])
    print(f"\n  Final output:")
    print(f"    MAE     = {m_out['mae']:.4f}")
    print(f"    MaxErr  = {m_out['max_err']:.4f}")
    print(f"    Corr    = {m_out['corr']:.4f}")

print(f"\n{'='*90}")
print(f"  ◄  = error tăng so với layer trước")
print(f"  ◄◄ = error tăng mạnh (>1.5×)")
print(f"{'='*90}")